Some images have a blue hue. How many are there and should we convert them to greyscale?

In [4]:
import os
from PIL import Image
import numpy as np
import pandas as pd
MAIN_DIR = "/home/onuro/code/simonwilliams32/MRI_project"
BASE_DIR = os.path.join(MAIN_DIR, "raw_data", "final_brain_tumor_preprocessed_dataset")
IMAGE_EXTS = (".png", ".jpg", ".jpeg")

def is_grayscale(path):
    """Returns True if image is black & white (all RGB channels equal)."""
    try:
        img = Image.open(path).convert("RGB")
        arr = np.array(img)
        return np.array_equal(arr[:, :, 0], arr[:, :, 1]) and np.array_equal(arr[:, :, 1], arr[:, :, 2])
    except Exception as e:
        print(f"Error reading {path}: {e}")
        return None

results = []

for split in os.listdir(BASE_DIR):
    split_path = os.path.join(BASE_DIR, split)
    if not os.path.isdir(split_path):
        continue

    for class_name in os.listdir(split_path):
        class_path = os.path.join(split_path, class_name)
        if not os.path.isdir(class_path):
            continue

        for fname in os.listdir(class_path):
            if not fname.lower().endswith(IMAGE_EXTS):
                continue
            fpath = os.path.join(class_path, fname)
            gray = is_grayscale(fpath)
            results.append({
                "split": split,
                "class": class_name,
                "filename": fname,
                "path": fpath,
                "is_grayscale": gray
            })

df = pd.DataFrame(results)

print(f"Total images checked: {len(df)}")
print(f"Grayscale: {(df['is_grayscale'] == True).sum()}")
print(f"Colormapped/Color: {(df['is_grayscale'] == False).sum()}")
print(f"Errors/unreadable: {df['is_grayscale'].isna().sum()}")

print("\nBreakdown by split and class:")
summary = df.groupby(["split", "class"])["is_grayscale"].value_counts().unstack(fill_value=0)
print(summary)

Total images checked: 14602
Grayscale: 11406
Colormapped/Color: 3196
Errors/unreadable: 0

Breakdown by split and class:
is_grayscale             False  True 
split        class                   
external_val glioma          0   2004
             meningioma      0   2004
test         glioma        223    338
             meningioma    102    263
             notumor        17    243
             pituitary     132    272
train        glioma        993   1623
             meningioma    493   1208
             notumor       102   1110
             pituitary     663   1223
val          glioma        210    351
             meningioma    113    251
             notumor        13    247
             pituitary     135    269
